<a href="https://colab.research.google.com/github/raddified/Vehicle_Damage_Severity_Classification/blob/develop/Model_Development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torchvision
import albumentations
import wandb

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("issamjebnouni/cardd")

print("Path to dataset files:", path)

100%|██████████| 2.81G/2.81G [00:37<00:00, 79.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/issamjebnouni/cardd/versions/1


# TASK 3: The Ordinal Rubric Script

In [ ]:
def map_to_ordinal_rank(label_name: str, bbox: list, img_width: int, img_height: int) -> int:
    """
    Maps categorical CarDD labels to Ordinal Severity Ranks (1=Minor, 2=Moderate, 3=Severe).
    bbox format assumed: [x_min, y_min, x_max, y_max]
    """
    label = label_name.lower()

    # Rank 1: Minor (Surface level)
    if label in ['scratch', 'crack']:
        return 1

    # Rank 3: Severe (Safety/Structural)
    if label in ['glass shatter', 'tire flat']:
        return 3

    # Rank 2 or 3: Dents and Broken Lamps
    if label in ['dent', 'lamp broken']:
        # Calculate bounding box area
        bbox_width = bbox[2] - bbox[0]
        bbox_height = bbox[3] - bbox[1]
        bbox_area = bbox_width * bbox_height
        image_area = img_width * img_height

        # If a dent covers more than 30% of the image, upgrade to Severe
        if (bbox_area / image_area) > 0.30:
            return 3
        return 2

    # Fallback for unknown classes
    return 1

# TASK 4: Image Preprocessing Pipeline

In [ ]:
def get_stage1_transforms(img_size=640):
    """
    Returns the Albumentations transformation pipeline for Stage 1 Object Detection.
    Crucially includes Letterbox padding to preserve aspect ratio.
    """
    return A.Compose([
        # 1. Resize the longest side to max_size, keeping aspect ratio intact
        A.LongestMaxSize(max_size=img_size),
        # 2. Pad the shorter side with neutral gray (128) to make it a perfect square
        A.PadIfNeeded(min_height=img_size, min_width=img_size,
                      border_mode=cv2.BORDER_CONSTANT, value=[128, 128, 128]),
        # 3. Normalize using ImageNet standard stats
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        # 4. Convert to PyTorch Tensor format (Channels First: C, H, W)
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

# TASK 5: PyTorch Dataset Class

In [ ]:
class CarDDStage1Dataset(Dataset):
    def __init__(self, image_dir, annotation_file, transforms=None):
        self.image_dir = image_dir
        self.transforms = transforms

        print(f"Loading annotations from {annotation_file}...")
        with open(annotation_file, 'r') as f:
            self.coco_data = json.load(f)

        # Parse COCO format into efficient lookup dictionaries
        self.images = {img['id']: img for img in self.coco_data['images']}
        self.categories = {cat['id']: cat['name'] for cat in self.coco_data['categories']}

        self.annotations = {}
        for ann in self.coco_data['annotations']:
            img_id = ann['image_id']
            if img_id not in self.annotations:
                self.annotations[img_id] = []
            self.annotations[img_id].append(ann)

        self.image_ids = list(self.images.keys())
        print(f"Successfully loaded {len(self.image_ids)} images.")

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.images[img_id]
        img_path = os.path.join(self.image_dir, img_info['file_name'])

        # Read Image using OpenCV
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Image not found at {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        img_height, img_width = image.shape[:2]

        bboxes = []
        labels = []

        # Extract and format bounding boxes if they exist for this image
        if img_id in self.annotations:
            for ann in self.annotations[img_id]:
                # COCO format: [x_min, y_min, width, height]
                x_min, y_min, w, h = ann['bbox']

                # Convert to Pascal VOC format: [x_min, y_min, x_max, y_max] required by Albumentations
                x_max = x_min + w
                y_max = y_min + h

                # Retrieve category name and apply Ordinal Rubric
                cat_name = self.categories[ann['category_id']]
                ordinal_rank = map_to_ordinal_rank(cat_name, [x_min, y_min, x_max, y_max], img_width, img_height)

                bboxes.append([x_min, y_min, x_max, y_max])
                labels.append(ordinal_rank)

        # Apply transforms (Padding, Scaling, Normalization)
        if self.transforms and len(bboxes) > 0:
            transformed = self.transforms(image=image, bboxes=bboxes, class_labels=labels)
            image = transformed['image']
            bboxes = transformed['bboxes']
            labels = transformed['class_labels']

        # If no bounding boxes, just transform the image
        elif self.transforms and len(bboxes) == 0:
            transformed = self.transforms(image=image)
            image = transformed['image']

        # Construct PyTorch target dictionary
        target = {}
        target["boxes"] = torch.tensor(bboxes, dtype=torch.float32) if len(bboxes) > 0 else torch.zeros((0, 4), dtype=torch.float32)
        target["labels"] = torch.tensor(labels, dtype=torch.int64) if len(labels) > 0 else torch.zeros((0,), dtype=torch.int64)
        target["image_id"] = torch.tensor([img_id])

        return image, target



# Testing & Visualization Helper

In [ ]:
def visualize_sample(image_tensor, target_dict):
    """
    Un-normalizes the PyTorch tensor and plots the image with bounding boxes and Ordinal Ranks.
    """
    # Define colors for ranks (1=Green, 2=Orange, 3=Red)
    color_map = {1: 'lime', 2: 'orange', 3: 'red'}
    rank_names = {1: 'Minor', 2: 'Moderate', 3: 'Severe'}

    # Un-normalize image for matplotlib viewing
    image_np = image_tensor.permute(1, 2, 0).numpy() # Change (C,H,W) to (H,W,C)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image_np = std * image_np + mean
    image_np = np.clip(image_np, 0, 1)

    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(image_np)

    boxes = target_dict['boxes'].numpy()
    labels = target_dict['labels'].numpy()

    for box, label in zip(boxes, labels):
        x_min, y_min, x_max, y_max = box
        width, height = x_max - x_min, y_max - y_min

        color = color_map.get(label, 'white')
        text = f"{rank_names.get(label, 'Unknown')} (Rank {label})"

        # Draw bounding box
        rect = patches.Rectangle((x_min, y_min), width, height, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)

        # Draw label text
        ax.text(x_min, y_min - 5, text, color='white', fontsize=12, fontweight='bold', bbox=dict(facecolor=color, alpha=0.8, pad=2))

    plt.axis('off')
    plt.title("Stage 1 Dataloader Test: Preprocessed Image + Ordinal Ranks")
    plt.show()

if __name__ == "__main__":
    # --- Instructions to Run ---
    # 1. Update these paths to point to your unzipped CarDD dataset in Google Colab/Local Drive.
    # Typically, the images are in a folder and annotations are in a COCO formatted .json file.
    IMAGE_DIR = "/content/dataset/CarDD/images/train"
    ANNOTATION_FILE = "/content/dataset/CarDD/annotations/instances_train.json"

    # 2. Check if paths exist before running
    if os.path.exists(IMAGE_DIR) and os.path.exists(ANNOTATION_FILE):
        print("Dataset found! Initializing DataLoader...")

        # Create dataset instance
        transforms = get_stage1_transforms(img_size=640)
        dataset = CarDDStage1Dataset(IMAGE_DIR, ANNOTATION_FILE, transforms=transforms)

        # Create DataLoader (collate_fn is required for object detection because images have varying numbers of boxes)
        def collate_fn(batch):
            return tuple(zip(*batch))

        dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

        # Test pulling one batch and visualizing the first image
        images, targets = next(iter(dataloader))
        print(f"Batch loaded! Image shape: {images[0].shape}, Bounding boxes found: {len(targets[0]['boxes'])}")

        visualize_sample(images[0], targets[0])
    else:
        print(f"Please update IMAGE_DIR and ANNOTATION_FILE paths to match your dataset location.")